In [ ]:
%pip install                                              \
        numpy                 adlfs               httpx   \
        python-dotenv         openai              polars  \
		azure-ai-inference    packaging

%restart_python

In [ ]:
import os
import asyncio
import json
import httpx
import numpy  as np
import scipy  as sp
from tqdm         import tqdm
from tqdm.asyncio import tqdm as asynctqdm
from dotenv       import load_dotenv
from openai       import OpenAI, AsyncOpenAI

import polars as pl
import matplotlib.pyplot as plt

import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from delta       import configure_spark_with_delta_pip

from azure.ai.inference.models import SystemMessage, UserMessage

from llm_agent   import LlmAgent


In [ ]:
builder = SparkSession.builder\
            .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")\
            .config("spark.sql.parquet.output.committer.class", "org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter")\
            .config("spark.mapreduce.fileoutputcommitter.marksuccessfuljobs","false")\
            .config("spark.sql.adaptive.enabled", True)\
            .config("spark.sql.shuffle.partitions", "auto")\
            .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "100MB")\
            .config("spark.sql.adaptive.coalescePartitions.enabled", True)\
            .config("spark.sql.dynamicPartitionPruning.enabled", True)\
            .config("spark.sql.autoBroadcastJoinThreshold", "10MB")\
            .config("spark.sql.session.timeZone", "Asia/Tokyo")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
            .config("spark.databricks.delta.write.isolationLevel", "SnapshotIsolation")\
            .config("spark.databricks.delta.optimizeWrite.enabled", True)\
            .config("spark.databricks.delta.autoCompact.enabled", True)
            # Delta Lake 用の SQL コミットプロトコルを指定
            # Parquet 出力時のコミッタークラスを指定
            # Azure Blob Storage (ABFS) 用のコミッターファクトリを指定
            # '_SUCCESS'で始まるファイルを書き込まないように設定
            # AQE(Adaptive Query Execution)の有効化
            # パーティション数を自動で調整するように設定
            # シャッフル後の1パーティションあたりの最小サイズを指定
            # AQEのパーティション合成の有効化
            # 動的パーティションプルーニングの有効化
            # 小さいテーブルのブロードキャスト結合への自動変換をするための閾値調整
            # SparkSessionのタイムゾーンを日本標準時刻に設定
            # Delta Lake固有のSQL構文や解析機能を拡張モジュールとして有効化
            # SparkカタログをDeltaLakeカタログへ変更
            # Delta Lake書き込み時のアイソレーションレベルを「スナップショット分離」に設定
            # 書き込み時にデータシャッフルを行い、大きなファイルを生成する機能の有効化
            # 書き込み後に小さなファイルを自動で統合する機能の有効化


spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
# 例
# BRAND_NAME_FILE = '遊戯王'

# 手入力
BRAND_NAME_FILE = ''

In [ ]:
# .env ファイルを読み込む
load_dotenv()

# 環境変数の取得
AI_FOUNDRY_ENDPOINT       = os.environ.get("AI_FOUNDRY_ENDPOINT")
AI_FOUNDRY_API_KEY        = os.environ.get("AI_FOUNDRY_API_KEY")
AI_FOUNDRY_MODEL          = os.environ.get("AI_FOUNDRY_MODEL")
LLM_MAX_TOKENS            = os.environ.get("LLM_MAX_TOKENS")
LLM_TEMPERATURE           = os.environ.get("LLM_TEMPERATURE")
LLM_TOP_P                 = os.environ.get("LLM_TOP_P")


# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
LLM_MAX_TOKENS       = int(LLM_MAX_TOKENS)
LLM_TEMPERATURE      = float(LLM_TEMPERATURE)
LLM_TOP_P            = float(LLM_TOP_P)

# 簡易デバッグ用
# print(f'AI_FOUNDRY_ENDPOINT:       {AI_FOUNDRY_ENDPOINT}')       # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_API_KEY:        {AI_FOUNDRY_API_KEY}')        # セキュリティリスクのため、コメントアウト
print(f'AI_FOUNDRY_MODEL:          {AI_FOUNDRY_MODEL}')
print(f'LLM_MAX_TOKENS:            {LLM_MAX_TOKENS}')
print(f'LLM_TEMPERATURE:           {LLM_TEMPERATURE}')
print(f'LLM_TOP_P:                 {LLM_TOP_P}')

In [ ]:
limits      = httpx.Limits(max_keepalive_connections=20, max_connections=300)
timeout     = httpx.Timeout(300.0, connect=5.0)
http_client = httpx.AsyncClient(limits=limits, timeout=timeout)
openai_cli  = AsyncOpenAI(
                    base_url=AI_FOUNDRY_ENDPOINT,
                    api_key=AI_FOUNDRY_API_KEY,
                    http_client=http_client,
                    max_retries=3
                )
semaphore   = asyncio.Semaphore(50)
llmClient   = LlmAgent(openai_cli, AI_FOUNDRY_MODEL, int(LLM_MAX_TOKENS), float(LLM_TEMPERATURE), float(LLM_TOP_P), semaphore)

with open(f'ブランドLP/{BRAND_NAME_FILE}', 'r', encoding='utf-8') as f:
    res_str = f.read()
print(res_str)

In [ ]:
# 特定の地点に対するADIDリストを取得
POLYGONLIST_TABLE = "adinte_adintedmp.envprod.polygonlist"
sdf_polygonlist   = spark.read.table(POLYGONLIST_TABLE)\
							.select(['project', 'caption', 'extraction_id', 'branch_id', 'spot_id'])\
                            .filter(col('extraction_id') == 44107)
ADIDLIST_TABLE    = "adinte_adintedmp.envprod.source"
sdf_adidlist      = spark.read.table(ADIDLIST_TABLE)\
							.select(['extraction_id', 'branch_id', 'spot_id', 'adid'])\
                            .filter(col('extraction_id') == 44107)\
							.join(sdf_polygonlist, on=['extraction_id', 'branch_id', 'spot_id'], how='inner')\
                            .select(['caption', 'adid'])\
                            .dropDuplicates()

target_captionlist = [row['caption'] for row in sdf_adidlist.select('caption').dropDuplicates().collect()]
target_adidlist    = [row['adid']    for row in sdf_adidlist.select('adid').dropDuplicates().collect()]
pdf_adidlist       = sdf_adidlist.toPandas()
pdf_adidlist

In [ ]:
# メモ：
# cohort.npzは以下のようなデータ構成になっている
# cohort.npz
#     |-- data              : 計算済みコホート係数行列
#     |-- indices           : データの位置指定子(列)
#     |-- indptr            : 行ごとのスライスインデックス
#     |-- shape             : コホート係数行列の形(ADIDのリスト × コホートキャプションのリスト)
#     |-- adid_list         : ADIDのリスト(列)
#     |-- business_codelist : コホートキャプションIDのリスト(行)
TARGET      = "/Volumes/stgadintedmpadintedi/featurestore/behaviorvector/cohort.npz"
npz         = np.load(TARGET, allow_pickle=True)
np_cohort   = sp.sparse.csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
np_adidlist = npz["adid_list"]

# 特定のADIDのみの抜き出し
indexer     = pd.Index(np_adidlist)
indices     = indexer.get_indexer(target_adidlist)
indices     = indices[indices != -1]
np_cohort   = np_cohort[indices, :]
np_adidlist = np_adidlist[indices]


# メモ：
# L2正規化を行うにあたって、疎行列専用に書く必要がある
# 密行列であれば簡単に書くことが可能であるが、今回の要件に適合しない
# そのため疎行列な対角行列を経由することにより、これを実現することとした
# 
# 当初すでにL2正規化は適用済みであったが、時々適用されていない状態になることがあるようだ
# そのため、改めて適用することとした
# L2正規化
if not np.allclose(np_cohort.power(2).sum(axis=1), 1.0):
	l2norm_vector = sp.sparse.linalg.norm(np_cohort, axis=1)
	np_cohort     = sp.sparse.diags(1 / np.maximum(l2norm_vector, 1e-10)) @ np_cohort


# CODELISTに対応するキャプションの文脈行列を取得
CAPTION_CONTEXT_MATRIX = "cohort_caption_matrix.npz"
npz                    = np.load(CAPTION_CONTEXT_MATRIX, allow_pickle=True)
relational_spots       = npz["business_placelist"]

# 簡易チェック
assert np.allclose(np_cohort.power(2).sum(axis=1), 1.0), "Not all row vectors have a unit length (squared norm != 1.0)."

print(np_cohort.shape)

In [ ]:
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視


# 特定のADIDに対する性別・年齢層を取得
window_spec      = Window.partitionBy('caption')
AGE_GENDER_TABLE = "adinte_datainfrastructure.master.mobilewalla_agegender"
sdf_agegender    = spark.read.table(AGE_GENDER_TABLE)\
						.select( ['adid', 'age', 'gender'])\
                        .join(sdf_adidlist, on='adid', how='inner')\
                        .select( ['caption', 'adid', 'age', 'gender'])\
                        .groupBy(['caption', 'age', 'gender'])\
                        .agg(F.count('*').alias('count'))\
                        .withColumn(
							'probability', 
							F.col('count') / F.sum('count').over(window_spec)
						)\
                        .select( ['caption', 'age', 'gender', 'probability'])\
                        .withColumnRenamed('caption', 'target')\
                        .select( ['target', 'age', 'gender', 'probability'])
OUTPUT_PATH      = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/preprocess/{BRAND_NAME_FILE}"
sdf_agegender.coalesce(1).write.mode("overwrite").parquet(OUTPUT_PATH)
pldf_agegender   = pl.read_parquet(OUTPUT_PATH+"/*.parquet")


# 特定のADIDに対する行動ベクトルを取得
tmp_cohort       = np_cohort.tocoo()
joined_spots     = np.array(['_'.join(spot) for spot in relational_spots])
pldf_behavioral  = pl.DataFrame({
    						'adid'            : np_adidlist[ tmp_cohort.row], 
                            'cohort_caption'  : joined_spots[tmp_cohort.col], 
                            'probability'     : tmp_cohort.power(2).data
                        })\
						.cast({
                            'adid'            : pl.String,
                            'cohort_caption'  : pl.String,
                            'probability'     : pl.Float32,
                        })
pldf_behavioral  = pl.from_pandas(pdf_adidlist)\
						.join(pldf_behavioral, on='adid', how='inner')\
                        .select(['caption', 'adid', 'cohort_caption', 'probability'])\
                        .rename({'caption': 'target'})\
                        .select( ['target', 'adid', 'cohort_caption', 'probability'])\
                        .group_by(['target', 'cohort_caption'])\
                        .agg(pl.col('probability').mean().alias('probability'))\
                        .select( ['target', 'cohort_caption', 'probability'])
pldf_behavioral

In [ ]:

# 目標の場所毎にAPIを投げる
list_tasks = []
for target_caption in tqdm(target_captionlist):
	tmp_agegender  = pldf_agegender.filter( pl.col('target') == target_caption)
	tmp_behavioral = pldf_behavioral.filter(pl.col('target') == target_caption)

	# データフレームの整形
	tmp_agegender  = tmp_agegender\
						.with_columns(
							age_gender_str=("'" + pl.col("age").cast(pl.String) + "' × '" + pl.col("gender").cast(pl.String) + "'")
						)

	# リスト形式への変換
	list_agegender  = list(tmp_agegender.select( ['age_gender_str', 'probability']).iter_rows())
	list_behavioral = list(tmp_behavioral.select(['cohort_caption', 'probability']).iter_rows())

	messages  = []
	messages.append(SystemMessage(content=(
					"# 前提条件\n"
					"あなたは外資系コンサルティングファーム出身の、極めて優秀なプロダクトマーケティングマネージャー兼リテール戦略アナリストです。\n"
					"ある特定のターゲット地点（店舗やエリア）において、【特定の商品】が販売されています。\n"
					"この地点に訪れる顧客の「属性データ」と「行動（回遊）データ」を提供します。\n"
					"商品の持つ特性と、実際のリアルなデータを高度に統合し、ターゲット地点においてどのような商戦・マーケティングを展開すべきか、具体的なアクションプランを提案してください。\n\n"

					"# データの定義\n"
					"# 入力情報\n"
					f"1. 【販売中のコア商材】: {BRAND_NAME_FILE}\n\n"
					
					"2. 【属性データ】(targetの客層構成)\n"
					"- target: 分析対象の場所\n"
					"- age: 年齢層\n"
					"- gender: 性別\n"
					"- probability: targetにおけるその属性（年齢×性別）の構成比・出現確率\n\n"

					"3. 【行動・回遊データ】(targetの顧客が他にどこに行っているか)\n"
					"- target: 分析対象の場所\n"
					"- cohort_caption: targetに来訪した顧客が、他に訪れている別の場所・施設（競合や親和性の高いスポット）\n"
					"- probability: targetの顧客が、そのcohort_captionを訪れる確率・親和度スコア\n\n"

					"# 実際の抽出データ\n"
					"【属性データ】\n"
					f"{list_agegender}\n\n"

					"【行動・回遊データ】\n"
					f"{list_behavioral}\n\n"

					"# 指示事項\n"
					"提供されたデータに基づき、以下の構成で詳細な分析レポートと戦略提案を出力してください。\n\n"

					"## 1. 顧客ペルソナとカスタマーの深い洞察\n"
					"- 属性と「他にどこに行っているか（cohort_caption）」を掛け合わせ、ターゲット地点に来訪するメイン客層の具体的なライフスタイル、価値観、来店目的を推測してください。\n"
					"- ターゲット地点とcohort_captionをどのように行き来しているか、顧客の1日の行動導線（カスタマー）の仮説を立ててください。\n"

					"## 2. ターゲット地点が持つ「真の価値」とポジショニング\n"
					"- 顧客はなぜ数ある場所の中からターゲット地点を訪れるのか。cohort_captionとの比較や相関から、この場所が提供している独自の価値（強み）を言語化してください。\n\n"

					"## 3. 具体的な商戦・マーケティング戦略への提言（ネクストアクション）\n"
					"- **店舗レイアウト・MD提案:** 判明したペルソナと回遊目的を踏まえ、ターゲット地点でどのような商品展開、陳列、あるいは看板・POPのメッセージングを行うべきか。\n"
					"- **コラボ＆送客戦略:** cohort_caption（他の来訪スポット）と競合して奪い合うべきか、あるいは相乗効果を狙って連携・クロスセルを図るべきか。具体的な施策案を提示してください。\n"
					"- **広告・販促アプローチ:** この客層に対して、いつ、どのタイミングで、どのようなプロモーションを打つのが最も投資対効果が高いか提案してください。\n"
				)))
	messages.append(UserMessage(content=res_str))
	list_tasks.append(llmClient.complete(messages))

analysis_datas = await asynctqdm.gather(*list_tasks)


# データフレームへの変換
pdf_final_analysis = pd.DataFrame([{
        'caption' : target_name,
        '分析結果'  : json.dumps(response_dict, indent=4, ensure_ascii=False),
    } for target_name, response_dict in zip(target_captionlist, analysis_datas)])

pdf_final_analysis

In [ ]:
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME_FILE}_analysis.parquet"
pdf_final_analysis.to_parquet(TARGET, compression="snappy")